# NLP Lab 01: Introduction to Natural Language Processing

## Setting Up NLP Libraries

We will use three libraries throughout this lab:

| Library | Best for | Language focus |
|---|---|---|
| **NLTK** (Natural Language Toolkit) | Teaching/classic NLP: tokenization, POS tagging, simple parsing, corpora | Mainly English |
| **spaCy** | Fast, production-grade pipelines: tokenization, POS, dependency parsing, NER | Many languages (via models), mainly English here |
| **PyThaiNLP** | Thai-specific NLP: word segmentation, POS tagging, NER, stopwords, transliteration | Thai |

Run the cell below once to install/download everything you need. If you're on a shared or offline machine, some downloads (like the spaCy model) may need to be run separately or may already be cached.

In [41]:
# Run this once. Uncomment if the packages are not already installed in your environment.
%pip install -q nltk spacy pythainlp scikit-learn
!python -m spacy download en_core_web_sm
print("If needed, uncomment the install lines above and re-run this cell.")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
If needed, uncomment the install lines above and re-run this cell.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import nltk

# Download the small set of NLTK data packages we need for this lab.
for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "stopwords", "maxent_ne_chunker", "maxent_ne_chunker_tab", "words"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Skipped {pkg}: {e}")

print("NLTK data ready.")

NLTK data ready.


## NLTK example (English)

NLTK gives us the classic NLP pipeline building blocks: tokenization, POS tagging, and a simple chunk-based NER.
- https://www.nltk.org//
- https://www.geeksforgeeks.org/nlp/nltk-tutorial/ 


In [32]:
from nltk.tokenize import word_tokenize
from nltk import pos_tag, ne_chunk
from nltk.corpus import stopwords

sentence = "Apple is looking at buying a U.K. startup for $1 billion."

# 1. Tokenization: split raw text into words/punctuation tokens
tokens = word_tokenize(sentence)
print("Tokens:", tokens)

# 2. Part-of-speech (POS) tagging: label each token's grammatical role
tagged = pos_tag(tokens)
print("\nPOS tags:", tagged)

# 3. Named Entity Recognition (a simple, rule/statistics-based chunker)
tree = ne_chunk(tagged)
print("\nNamed entities:")
for subtree in tree:
    if hasattr(subtree, "label"):
        entity_text = " ".join(word for word, tag in subtree.leaves())
        print(f"  {entity_text}  ->  {subtree.label()}")

# 4. Stopword removal (common, low-information words)
stop_words = set(stopwords.words("english"))
filtered = [w for w in tokens if w.lower() not in stop_words and w.isalpha()]
print("\nTokens without stopwords:", filtered)

Tokens: ['Apple', 'is', 'looking', 'at', 'buying', 'a', 'U.K.', 'startup', 'for', '$', '1', 'billion', '.']

POS tags: [('Apple', 'NNP'), ('is', 'VBZ'), ('looking', 'VBG'), ('at', 'IN'), ('buying', 'VBG'), ('a', 'DT'), ('U.K.', 'NNP'), ('startup', 'NN'), ('for', 'IN'), ('$', '$'), ('1', 'CD'), ('billion', 'CD'), ('.', '.')]

Named entities:
  Apple  ->  GPE

Tokens without stopwords: ['Apple', 'looking', 'buying', 'startup', 'billion']


## spaCy example (English)

spaCy bundles tokenization, POS tagging, dependency parsing, and NER into a single, fast pipeline (`nlp(text)`).
- https://spacy.io/
- https://www.tutorialspoint.com/spacy/index.htm

In [33]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    raise RuntimeError(
        "Model not found. Run: python -m spacy download en_core_web_sm"
    )

doc = nlp("Apple is looking at buying a U.K. startup for $1 billion.")

print(f"{'TOKEN':<12}{'POS':<8}{'DEP':<12}{'HEAD'}")
for token in doc:
    print(f"{token.text:<12}{token.pos_:<8}{token.dep_:<12}{token.head.text}")

print("\nNamed entities:")
for ent in doc.ents:
    print(f"  {ent.text:<20} -> {ent.label_}")

TOKEN       POS     DEP         HEAD
Apple       PROPN   nsubj       looking
is          AUX     aux         looking
looking     VERB    ROOT        looking
at          ADP     prep        looking
buying      VERB    pcomp       at
a           DET     det         U.K.
U.K.        PROPN   dobj        buying
startup     NOUN    advcl       looking
for         ADP     prep        startup
$           SYM     quantmod    billion
1           NUM     compound    billion
billion     NUM     pobj        for
.           PUNCT   punct       looking

Named entities:
  Apple                -> ORG
  U.K.                 -> GPE
  $1 billion           -> MONEY


## PyThaiNLP example (Thai)

PyThaiNLP handles the Thai-specific problem we introduced in Section 1: **there are no spaces between Thai words**, so tokenization requires a dedicated algorithm/model rather than simply splitting on whitespace.
- https://pythainlp.org/

In [34]:
from pythainlp.tokenize import word_tokenize as th_word_tokenize
from pythainlp.tag import pos_tag as th_pos_tag
from pythainlp.corpus import thai_stopwords

thai_sentence = "ฉันชอบกินข้าวผัดกะเพราที่ร้านนี้มากเลยครับ"

# 1. Word tokenization (default engine: "newmm", a dictionary-based maximal matching + TCC)
th_tokens = th_word_tokenize(thai_sentence)
print("Thai tokens:", th_tokens)

# 2. POS tagging
th_tagged = th_pos_tag(th_tokens)
print("\nThai POS tags:", th_tagged)

# 3. Stopword removal
th_stopwords = thai_stopwords()
th_filtered = [w for w in th_tokens if w not in th_stopwords and w.strip() != ""]
print("\nThai tokens without stopwords:", th_filtered)

Thai tokens: ['ฉัน', 'ชอบ', 'กินข้าว', 'ผัดกะเพรา', 'ที่', 'ร้าน', 'นี้', 'มาก', 'เลย', 'ครับ']

Thai POS tags: [('ฉัน', 'PPRS'), ('ชอบ', 'VACT'), ('กินข้าว', 'NCMN'), ('ผัดกะเพรา', 'NCMN'), ('ที่', 'PREL'), ('ร้าน', 'NCMN'), ('นี้', 'DDAC'), ('มาก', 'ADVN'), ('เลย', 'ADVN'), ('ครับ', 'ADVN')]

Thai tokens without stopwords: ['ชอบ', 'กินข้าว', 'ผัดกะเพรา', 'ร้าน']


## Core Challenge Demo: Thai Word Segmentation Ambiguity

Let's revisit the ambiguous sentence from Section 1 and see how different tokenization **engines** inside PyThaiNLP can actually produce different segmentations — this is a live demonstration of why word segmentation is treated as its own hard NLP research problem for Thai, rather than a "solved" preprocessing step.

In [35]:
from pythainlp.tokenize import word_tokenize

sentences = [
    "อาจารย์สอนภาษาไทยให้กับนักเรียนชาติจีน",
    "เขาตัดทรงผมใหม่แล้วดูดีมาก",
    
]

for sent in sentences:
    print(f"ประโยค: {sent}")
    for engine in ["newmm", "longest", "mm"]:
        try:
            result = word_tokenize(sent, engine=engine)
            print(f"  {engine:>10}: {result}")
        except Exception as e:
            print(f"  {engine:>10}: Error ({e})")
    print("-" * 40)


ประโยค: อาจารย์สอนภาษาไทยให้กับนักเรียนชาติจีน
       newmm: ['อาจารย์', 'สอน', 'ภาษาไทย', 'ให้', 'กับ', 'นักเรียน', 'ชาติ', 'จีน']
     longest: ['อาจารย์', 'สอน', 'ภาษาไทย', 'ให้', 'กับ', 'นักเรียน', 'ชาติ', 'จีน']
          mm: ['อาจารย์', 'สอนภาษาไทย', 'ให้', 'กับ', 'นักเรียน', 'ชาติ', 'จีน']
----------------------------------------
ประโยค: เขาตัดทรงผมใหม่แล้วดูดีมาก
       newmm: ['เขา', 'ตัด', 'ทรงผม', 'ใหม่', 'แล้ว', 'ดู', 'ดีมาก']
     longest: ['เขา', 'ตัด', 'ทรงผม', 'ใหม่', 'แล้ว', 'ดูดี', 'มาก']
          mm: ['เขา', 'ตัด', 'ทรงผม', 'ใหม่', 'แล้ว', 'ดูดีมาก']
----------------------------------------


## NLP Applications

### Text Classification

**Text classification** assigns a predefined label to a piece of text — e.g., sentiment (positive/negative), topic, spam/not-spam, or intent. It is a classic **NLU** task: text goes in, a structured label comes out.

Below is a minimal pipeline: turn text into numeric features (bag-of-words), then train a simple classifier. In real systems, the "text → numeric features" step is often replaced by upstream-learned embeddings (Word2Vec, BERT), but the classification idea stays the same.

In [46]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# Toy training data: (review, label) where 1 = positive, 0 = negative
train_texts = [
    "I love this movie, it was fantastic",
    "What a wonderful and amazing film",
    "This was a terrible and boring movie",
    "I hated every minute of this film",
    "Absolutely brilliant, I really enjoyed it",
    "Awful acting, a complete waste of time",
]
train_labels = [1, 1, 0, 0, 1, 0]

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_texts)

clf = LogisticRegression()
clf.fit(X_train, train_labels)

test_texts = ["This film was absolutely wonderful", "A boring and terrible experience"]
X_test = vectorizer.transform(test_texts)
predictions = clf.predict(X_test)

for text, pred in zip(test_texts, predictions):
    print(f"{text!r:55} -> {'positive' if pred == 1 else 'negative'}")

'This film was absolutely wonderful'                    -> positive
'A boring and terrible experience'                      -> negative


In [ ]:
# Same idea, but for Thai text: we swap the tokenizer used by CountVectorizer
# for PyThaiNLP's word_tokenize, since Thai has no whitespace to split on.
from pythainlp.tokenize import word_tokenize as th_tokenize

th_train_texts = [
    "อาหารร้านนี้อร่อยมากแนะนำเลย",       # this restaurant's food is very delicious, recommended
    "บริการดีมากประทับใจสุดๆ",           # the service was great, very impressed
    "อาหารรสชาติแย่มากไม่อร่อยเลย",       # the food tasted very bad, not delicious at all
    "บริการห่วยแตกรอนานมากไม่ประทับใจ",   # terrible service, waited very long, not impressed
]
th_train_labels = [1, 1, 0, 0]

th_vectorizer = CountVectorizer(tokenizer=th_tokenize, token_pattern=None)
X_th_train = th_vectorizer.fit_transform(th_train_texts)

th_clf = LogisticRegression()
th_clf.fit(X_th_train, th_train_labels)

th_test_texts = ["ร้านนี้อาหารอร่อยประทับใจมาก", "บริการแย่มากไม่แนะนำเลย"]
X_th_test = th_vectorizer.transform(th_test_texts)
th_predictions = th_clf.predict(X_th_test)

for text, pred in zip(th_test_texts, th_predictions):
    print(f"{text:35} -> {'positive' if pred == 1 else 'negative'}")

อาหารก็พอใช้ได้แหละแต่ที่หลังไม่มากินอีกแล้ว -> negative
บริการแย่มากไม่แนะนำเลย             -> negative


## DIY Exercises

Now it's your turn. Each exercise has:

1. A **DIY** cell with `# TODO` markers for you to fill in.
2. An **Answer Key** cell right after it — try to solve the exercise first, then compare.

---

### Exercise 1 — Thai Tokenization & Counting

Using `pythainlp`, tokenize the sentence below and print:
1. The list of tokens.
2. The **number of unique tokens** (after removing Thai stopwords).

In [ ]:
# === DIY Exercise 1 ===
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords

sentence = "วันนี้อากาศดีมากฉันอยากไปเที่ยวทะเลกับเพื่อนๆของฉัน"

# TODO 1: tokenize 'sentence' into a list of words using word_tokenize
tokens = word_tokenize(sentence)

# TODO 2: build a set of Thai stopwords using thai_stopwords()
stopwords_set = thai_stopwords()

# TODO 3: build 'unique_content_tokens': the SET of tokens in 'tokens'
#         that are NOT in 'stopwords_set' and are not blank/whitespace
unique_content_tokens = {t for t in tokens if t not in stopwords_set and t.strip() != ""}

print("Tokens:", tokens)
print("Number of unique content tokens:", len(unique_content_tokens) if unique_content_tokens else "TODO not done yet")
print("Content tokens:", unique_content_tokens)

Tokens: ['วันนี้', 'อากาศ', 'ดีมาก', 'ฉัน', 'อยาก', 'ไปเที่ยว', 'ทะเล', 'กับ', 'เพื่อน', 'ๆ', 'ของ', 'ฉัน']
Number of unique content tokens: 5
Content tokens: {'ไปเที่ยว', 'ดีมาก', 'ทะเล', 'อากาศ', 'เพื่อน'}


---

### Exercise 2 — Simple English Stopword Filter + Word Frequency

Write a function `top_words(text, n)` that:
1. Tokenizes `text` with NLTK's `word_tokenize`.
2. Lowercases every token and keeps only alphabetic tokens (use `str.isalpha()`).
3. Removes English stopwords (`nltk.corpus.stopwords`).
4. Returns the `n` most common remaining words as a list of `(word, count)` tuples.

Hint: `collections.Counter` has a `.most_common(n)` method.

In [61]:
# === DIY Exercise 2 ===
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter

text = """
Natural language processing is a fascinating field. Natural language is complex,
ambiguous, and endlessly creative. Processing natural language well requires
understanding both language structure and meaning.
"""

def top_words(text, n):
    # TODO 1: tokenize 'text'
    tokens = word_tokenize(text)

    # TODO 2: lowercase + keep only alphabetic tokens
    words = [t.lower() for t in tokens if t.isalpha()]

    # TODO 3: remove English stopwords
    stop_words = stopwords.words("english")
    filtered = [w for w in words if w not in stop_words]

    # TODO 4: count and return the n most common (word, count) pairs
    counts = Counter(filtered)
    return counts.most_common(n)

result = top_words(text, 3)
print(result)

[('language', 4), ('natural', 3), ('processing', 2)]


---

### Exercise 3 — Named Entity Extraction with spaCy

Given the paragraph below, use spaCy to extract and print only the entities labeled **`PERSON`** and **`ORG`** (organization), in the format `"<text>  ->  <label>"`.

In [74]:
# === DIY Exercise 3 ===
import spacy

nlp = spacy.load("en_core_web_sm")

paragraph = (
    "Sundar Pichai, the CEO of Google, met with representatives from Microsoft "
    "and OpenAI last week in San Francisco to discuss AI safety."
)

doc = nlp(paragraph)

# TODO: loop over doc.ents and print only entities whose .label_ is
#       "PERSON" or "ORG", formatted as f"{ent.text}  ->  {ent.label_}"

#HINT:
for ent in doc.ents:
     if ent.label_ in ("PERSON", "ORG"):
         print(f"{ent.text}  ->  {ent.label_}")

Sundar Pichai  ->  PERSON
Google  ->  ORG
Microsoft  ->  ORG
